<!--    
    ================================================================================================
    
    Copyright (c) 2026 Ilya Snegov

    Licensed under the Apache License, Version 2.0 (the "License");
    you may not use this file except in compliance with the License.
    You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

    ================================================================================================
    notebooks/04-model-evaluation.ipynb
    ================================================================================================
-->

# **[Financial Machine Learning Laboratory](https://github.com/snegoviol/finml-lab) — Model Evaluation**

*A Jupyter notebook for evaluating the final model performance on the unseen test set, including metric calculation, prediction visualization, and feature importance analysis.*

## **I. Setup the environment**

### **Import required libraries**

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

from finml_lib import set_matplotlib_style

### **Apply global Matplotlib style for consistent visualization**

In [ ]:
set_matplotlib_style()

### **Load the test dataset**

In [ ]:
preprocessed_test_df = pd.read_parquet("./data/preprocessed_test.parquet")

X_TEST = preprocessed_test_df.drop(columns=['target'])
Y_TEST = preprocessed_test_df['target']

### **Load the serialized model**

In [ ]:
model = XGBRegressor()
model.load_model("./data/finml_gbdt_regressor.json")

## **II. Evaluate the model on the unseen test dataset**

### **Generate predictions**

In [ ]:
y_test_pred = model.predict(X_TEST)

### **Compute the primary evaluation metric (Spearman Rank Correlation) and compare it against Pearson correlation**

In [ ]:
spearman_corr, spearman_p = spearmanr(Y_TEST, y_test_pred)
pearson_corr, pearson_p = pearsonr(Y_TEST, y_test_pred)

correlation_df = pd.DataFrame({
    'Metric': pd.Series(
        [
            'Test set observations',
            'Spearman Rank Correlation',
            'Pearson Linear Correlation',
            'Rank vs Linear Gap'
        ], 
        dtype=object
    ),
    'Value': pd.Series(
        [
            len(Y_TEST),
            spearman_corr,
            pearson_corr,
            abs(spearman_corr - pearson_corr)
        ], 
        dtype=object
    ),
    'P-Value': pd.Series(
        [
            np.nan,
            spearman_p,
            pearson_p,
            np.nan
        ], 
        dtype='Float64'
    )
})

correlation_df

### **Compare the descriptive statistics of actual and predicted return distributions**

In [ ]:
y_test_pred_series = pd.Series(y_test_pred, index=Y_TEST.index)

comparison_df = pd.DataFrame({
    'Metric': pd.Series(
        [
            'Mean',
            'Std Dev',
            'Min',
            '25th Percentile',
            'Median',
            '75th Percentile',
            'Max',
            'Skewness',
            'Kurtosis (excess)'
        ], 
        dtype=object
    ),
    'Actual (Y_TEST)': pd.Series(
        [
            Y_TEST.mean(),
            Y_TEST.std(),
            Y_TEST.min(),
            Y_TEST.quantile(0.25),
            Y_TEST.median(),
            Y_TEST.quantile(0.75),
            Y_TEST.max(),
            Y_TEST.skew(),
            Y_TEST.kurtosis()
        ], 
        dtype='Float64'
    ),
    'Predicted (y_pred)': pd.Series(
        [
            y_test_pred_series.mean(),
            y_test_pred_series.std(),
            y_test_pred_series.min(),
            y_test_pred_series.quantile(0.25),
            y_test_pred_series.median(),
            y_test_pred_series.quantile(0.75),
            y_test_pred_series.max(),
            y_test_pred_series.skew(),
            y_test_pred_series.kurtosis()
        ], 
        dtype='Float64'
    )
})

comparison_df

### **Compare the actual returns across prediction deciles**

In [ ]:
strategy_df = pd.DataFrame({
    'actual': Y_TEST.values,
    'predicted': y_test_pred
}, index=Y_TEST.index)

strategy_df['decile'] = pd.qcut(strategy_df['predicted'], q=10, labels=False, duplicates='drop')

decile_list = []
count_list = []
mean_actual_list = []
ic_decile_list = []
cum_return_list = []

cumulative_return = 1.0

for decile in sorted(strategy_df['decile'].unique()):
    subset = strategy_df[strategy_df['decile'] == decile]
    count = len(subset)
    mean_actual = subset['actual'].mean()
    decile_ic = spearmanr(subset['actual'], subset['predicted']).statistic if count > 1 else np.nan
    cumulative_return *= (1 + mean_actual)

    decile_list.append(int(decile))
    count_list.append(count)
    mean_actual_list.append(mean_actual)
    ic_decile_list.append(decile_ic)
    cum_return_list.append(cumulative_return - 1)

decile_df = pd.DataFrame({
    'Decile': pd.Series(
        decile_list, 
        dtype='Int64'
    ),
    'Count': pd.Series(
        count_list, 
        dtype='Int64'
    ),
    'Mean Actual': pd.Series(
        mean_actual_list, 
        dtype='Float64'
    ),
    'IC (Decile)': pd.Series(
        ic_decile_list, 
        dtype='Float64'
    ),
    'Cum. Return': pd.Series(
        cum_return_list, 
        dtype='Float64'
    )
})

decile_df

### **Compare the cumulative trajectories of actual and predicted returns**

In [ ]:
cum_actual = Y_TEST.cumsum()
cum_pred = y_test_pred_series.cumsum()

plt.plot(
    cum_actual.index, 
    cum_actual, 
    label='Cumulative Actual Returns', 
    color='blue'
)

plt.plot(
    cum_pred.index, 
    cum_pred, 
    label='Cumulative Predicted Returns', 
    color='red', 
    linestyle='--'
)

plt.axhline(0, color='gray', linestyle='-')

plt.title('Cumulative Returns: Actual vs Predicted')
plt.xlabel('Time (Index)')
plt.ylabel('Cumulative Return')
plt.legend(loc='upper left')
plt.xticks(rotation=45)
plt.show()

### **Compare the temporal dynamics of actual and predicted returns**

*The high frequency of observations renders the full timeline as a dense mass. Use zoom and pan to inspect local patterns.*

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=Y_TEST.index,
    y=Y_TEST,
    mode='lines',
    name='Actual Returns',
    line=dict(color='blue', width=1),
    opacity=0.6
))

fig.add_trace(go.Scatter(
    x=Y_TEST.index,
    y=y_test_pred_series,
    mode='lines',
    name='Predicted Returns',
    line=dict(color='red', width=1.5, dash='dash'),
    opacity=0.8
))

fig.update_layout(
    title='Time Series Comparison: Actual vs Predicted Returns',
    xaxis_title='Time (Index)',
    yaxis_title='Return Value',
    legend=dict(x=0.01, y=0.99),
)

fig.show()

## **III. Analysis of out-of-sample predictive performance**

### **Overview**

The out-of-sample evaluation reveals a statistically significant but attenuated predictive signal. The model achieves a Spearman Rank Correlation (Information Coefficient) of $\rho_S = 0.0424$ ($p = 0.0021$) on the test set, representing a notable degradation from the validation peak of $\rho_S = 0.0765$. This generalization gap is characteristic of non-stationary financial time series, where regime shifts (e.g., the severe drawdown observed in August–September 2023) invalidate historical patterns. Crucially, the Spearman coefficient exceeds the Pearson linear correlation ($\rho_P = 0.0293$), yielding a rank-linear gap of $\Delta = 0.0130$. This indicates that the model captures non-linear rank-order dependencies more effectively than linear magnitude scaling. However, a severe structural discrepancy exists between the actual and predicted distributions: the model exhibits extreme variance shrinkage, with the predicted standard deviation ($\sigma_{pred} = 0.00011$) being approximately 36 times smaller than the actual standard deviation ($\sigma_{actual} = 0.00395$). Furthermore, the model fails to replicate the heavy-tailed nature of the target, producing a platykurtic distribution ($\kappa_{pred} = 0.86$) compared to the highly leptokurtic actual returns ($\kappa_{actual} = 6.94$).

### **Evaluation of predictive characteristics**

| Characteristic | Mechanism / Observation |
| :--- | :--- |
| **Rank-Linear Gap** | Spearman exceeds Pearson, confirming the capture of non-linear rank dependencies over linear scaling. |
| **Variance Shrinkage** | Predicted variance is heavily compressed ($\sigma_{pred} \ll \sigma_{actual}$), indicating extreme conservatism in magnitude forecasting. |
| **Tail Behavior** | The model fails to extrapolate heavy tails, producing a near-normal distribution ($\kappa \approx 0.86$) instead of the actual fat-tailed regime ($\kappa \approx 6.94$). |
| **Decile Monotonicity** | Mean actual returns generally increase across predicted deciles, validating the model's utility for cross-sectional sorting despite variance compression. |
| **Trajectory Divergence** | The cumulative predicted trajectory is overly smooth, failing to capture the sharp drawdowns and volatility clusters of the actual cumulative returns. |

### **Conclusion**

In the context of financial forecasting, evaluating model performance requires distinguishing between *magnitude prediction* and *rank prediction*. The severe variance shrinkage and the smooth, monotonically increasing cumulative trajectory demonstrate that the model is fundamentally unsuitable for predicting the absolute magnitude of returns. It acts as an overly conservative mean-reverter, entirely missing the structural breaks and volatility clustering inherent in the actual time series (e.g., the August–September drawdown). The failure to replicate the heavy tails ($\kappa_{pred} \ll \kappa_{actual}$) further confirms that the model cannot extrapolate extreme market shocks.

However, the model's utility is preserved in the domain of **cross-sectional ranking**. The statistically significant Spearman IC ($\rho_S = 0.0424$), combined with the positive rank-linear gap, proves that the model successfully extracts a weak but persistent alpha signal. This is empirically validated by the decile analysis: despite the variance compression, the mean actual returns exhibit a generally monotonic increase from the bottom decile (Decile 0: $-0.000322$) to the upper-middle deciles (Decile 8: $0.000273$). The slight non-monotonicity at the extreme upper tail (Decile 9: $0.000163$) is a typical artifact of extreme volatility in high-signal bins, rather than a failure of the underlying ranking logic. 

Therefore, the model must be strictly deployed as a **ranking mechanism** for long/short portfolio construction (e.g., sorting assets into quantiles) rather than as an absolute return forecaster. The generalization gap from the validation set ($0.0765 \rightarrow 0.0424$) is an acceptable and expected penalty for out-of-sample robustness in highly noisy, non-stationary crypto markets, confirming that the model has learned genuine market microstructure rather than memorizing historical noise.

## **IV. Feature importance analysis**

### **Prepare the feature importance data and visualization utility**

In [ ]:
feature_importance_df = pd.DataFrame({
    'feature': X_TEST.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False).reset_index(drop=True)

def plot_feature_importance(feature_importance_df: pd.DataFrame, n_features: int, is_top: bool) -> None:
    if is_top:
        features = feature_importance_df.head(n_features)
        title = f'Top {n_features} Feature Importances (Gain)'
    else:
        features = feature_importance_df.tail(n_features)
        title = f'Bottom {n_features} Feature Importances (Gain)'
    
    sns.barplot(
        data=features,
        x='importance',
        y='feature',
        color='blue'
    )
    
    plt.title(title)
    plt.xlabel('Normalized Importance')
    plt.ylabel('Feature')
    plt.show()

### **Visualize the top 20 features with the highest normalized gain**

In [ ]:
plot_feature_importance(feature_importance_df, 20, is_top=True)

### **Visualize the bottom 20 features with the lowest normalized gain**

In [ ]:
plot_feature_importance(feature_importance_df, 20, is_top=False)

### **Compute the feature importance concentration and signal distribution**

In [ ]:
cumulative_importance = feature_importance_df['importance'].cumsum()

concentration_df = pd.DataFrame({
    'Metric': pd.Series(
        [
            'Total features',
            'Zero-importance features',
            'Features explaining 50% of gain',
            'Features explaining 90% of gain',
            'Top-10 cumulative gain'
        ], 
        dtype=object
    ),
    'Value': pd.Series(
        [
            len(feature_importance_df),
            (feature_importance_df['importance'] == 0).sum(),
            (cumulative_importance < 0.50).sum() + 1,
            (cumulative_importance < 0.90).sum() + 1,
            feature_importance_df['importance'].head(10).sum()
        ], 
        dtype=object
    )
})

concentration_df

## **VII. Analysis of feature importance and signal distribution**

### **Overview**

The feature importance analysis reveals a highly democratic and distributed signal landscape, characteristic of a robust machine learning model operating in a low signal-to-noise environment. Out of 73 engineered features, the top 10 features collectively account for only 20.16% of the total normalized gain, while 28 features are required to explain 50% of the model's predictive power. This absence of a single dominant feature is a critical indicator of structural integrity, effectively ruling out data leakage or overfitting to a spurious, high-variance predictor. Furthermore, the model explicitly pruned 7 features to zero importance. These discarded features are predominantly ultra-rare binary candlestick patterns (e.g., `CDLSTICKSANDWICH`, `CDLXSIDEGAP3METHODS`, `CDL3INSIDE`), confirming that the regularization mechanism successfully suppressed statistical noise. The most influential features represent a heterogeneous mix of temporal encodings (`dayofweek_sin`, `month_sin`), short- and long-term momentum oscillators (`MFI_24`, `ROC_24`, `ROC_168`), and specific high-frequency candlestick formations (`CDLBELTHOLD`, `CDLHARAMI`).

### **Evaluation of feature characteristics**

| Characteristic | Mechanism / Observation |
| :--- | :--- |
| **Signal Democratization** | Top-10 features explain only $\approx 20.16\%$ of the total gain, preventing reliance on a single spurious predictor. |
| **Noise Pruning** | 7 ultra-rare binary candlestick patterns were assigned zero importance, effectively removing them from the decision trees. |
| **Temporal Seasonality** | Sinusoidal time encodings (`dayofweek_sin`, `hour_sin`) rank highly, indicating the model captures intraday and weekly market microstructure cycles. |
| **Momentum Heterogeneity** | The model integrates both short-term (`ROC_24`, `WILLR_24`) and long-term (`ROC_168`, `DX_168`) momentum, balancing noise and trend. |

### **Conclusion**

In quantitative finance, a highly concentrated feature importance distribution is often a red flag for data leakage or look-ahead bias, where a single variable artificially dominates the loss landscape. Conversely, the observed "flat" distribution — requiring 57 out of 73 features to capture 90% of the cumulative gain — demonstrates that the model operates as a true ensemble of weak learners. It aggregates subtle, multidimensional microstructural signals rather than extrapolating from a single dominant factor.

The explicit suppression of 7 rare candlestick patterns to zero importance validates the efficacy of the regularization hyperparameters (`reg_alpha`, `min_child_weight`). By ignoring these sparse, binary events, the model avoids overfitting to statistical anomalies. Meanwhile, the high ranking of sinusoidal temporal features (`dayofweek_sin`, `month_sin`) and volume-weighted momentum indicators (`MFI_24`) suggests that the model has successfully learned the underlying market microstructure, specifically the liquidity cycles and session-based volatility patterns inherent to the cryptocurrency market.

Therefore, the feature importance landscape confirms that the model's predictive capability is derived from a broad, economically interpretable consensus of technical and temporal indicators. This distributed reliance on weak signals is precisely what allows the model to maintain a statistically significant, albeit attenuated, Information Coefficient on the out-of-sample test set, ensuring that the learned patterns are generalizable rather than memorized.